# PCA של ספקטרים מלאכותיים בעזרת SciPy\n\nבמחברת זו ניצור נתונים שמזכירים מדידות ספקטרוסקופיות של תערובות. כל דוגמה תהיה ספקטרום שלם, כלומר הרבה נקודות בליעה באורכי גל שונים. לאחר מכן נשתמש ב־PCA כדי להציג את הדוגמאות בגרף דו־ממדי.\n\nהמטרה היא להבין איך אפשר לראות מבנה בנתונים עם הרבה עמודות.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import svd

## יצירת ספקטרים של חומרים טהורים\n\nניצור שלושה חומרים דמיוניים. לכל חומר יש ספקטרום בליעה עם פס אחד או שניים. נשתמש בפונקציה גאוסיאנית כדי לקבל פס חלק.

In [ ]:
def gaussian(x, center, width, height):
    return height * np.exp(-(x - center)**2 / (2 * width**2))

wavelength = np.linspace(400, 700, 301) # nm

spectrum_A = gaussian(wavelength, 470, 18, 1.0) + gaussian(wavelength, 610, 35, 0.25)
spectrum_B = gaussian(wavelength, 535, 24, 0.9)
spectrum_C = gaussian(wavelength, 650, 20, 0.8) + gaussian(wavelength, 430, 30, 0.35)

In [ ]:
plt.figure()
plt.plot(wavelength, spectrum_A, label='A')
plt.plot(wavelength, spectrum_B, label='B')
plt.plot(wavelength, spectrum_C, label='C')
plt.xlabel('wavelength (nm)')
plt.ylabel('absorbance')
plt.title('Pure spectra')
plt.legend()
plt.show()

## יצירת תערובות\n\nכעת ניצור הרבה דוגמאות. כל דוגמה תהיה תערובת של שלושת הספקטרים הטהורים, עם קצת רעש מדידה. אם $x_A$, $x_B$, ו־$x_C$ הם מקדמי התערובת, אז הספקטרום הוא:\n\n$$S_\mathrm{mix}(\lambda) = x_A S_A(\lambda) + x_B S_B(\lambda) + x_C S_C(\lambda) + \mathrm{noise}$$

In [ ]:
rng = np.random.default_rng(123)

n_samples = 80
fractions = rng.dirichlet([1.2, 1.2, 1.2], size=n_samples)
pure_spectra = np.vstack([spectrum_A, spectrum_B, spectrum_C])

# כל שורה היא דוגמה, וכל עמודה היא אורך גל
data = fractions @ pure_spectra
data += rng.normal(0.0, 0.015, size=data.shape)

In [ ]:
plt.figure()
for i in range(12):
    plt.plot(wavelength, data[i], alpha=0.7)

plt.xlabel('wavelength (nm)')
plt.ylabel('absorbance')
plt.title('Some mixture spectra')
plt.show()

כל ספקטרום הוא וקטור עם 301 ערכים. קשה להשוות 80 דוגמאות כאלה בעין. PCA מנסה למצוא צירים חדשים שמסכמים את רוב השונות בנתונים.

## הכנה ל־PCA\n\nלפני PCA מחסרים מכל עמודה את הממוצע שלה. כך אנחנו מנתחים את ההבדלים בין הדוגמאות, ולא את הספקטרום הממוצע עצמו.

In [ ]:
mean_spectrum = np.mean(data, axis=0)
centered_data = data - mean_spectrum

In [ ]:
plt.figure()
plt.plot(wavelength, mean_spectrum)
plt.xlabel('wavelength (nm)')
plt.ylabel('absorbance')
plt.title('Mean spectrum')
plt.show()

## חישוב PCA בעזרת SVD\n\nנשתמש ב־`scipy.linalg.svd`. אין צורך לזכור את כל האלגברה בשלב הזה. מבחינה מעשית, הפקודה מוצאת כיוונים חשובים בנתונים. שני הכיוונים הראשונים יתנו לנו גרף דו־ממדי של הדוגמאות.

In [ ]:
U, s, Vt = svd(centered_data, full_matrices=False)

scores = U[:, :2] * s[:2]
explained_variance = s**2 / np.sum(s**2)

print(f'PC1 explains {100 * explained_variance[0]:.1f}% of the variance')
print(f'PC2 explains {100 * explained_variance[1]:.1f}% of the variance')

In [ ]:
plt.figure()
plt.scatter(scores[:, 0], scores[:, 1])
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA of mixture spectra')
plt.show()

הנקודות בגרף הן הדוגמאות המקוריות. כל דוגמה הייתה ספקטרום עם 301 נקודות, ועכשיו היא מיוצגת על־ידי שני מספרים בלבד. אם יש מבנה בתערובות, למשל דוגמאות עשירות בחומר מסוים, לעיתים אפשר לראות אותו בגרף הזה.

## צביעה לפי הרכב התערובת\n\nבנתונים ניסיוניים אמיתיים בדרך כלל לא נדע את ההרכב המדויק. כאן, מפני שאנחנו יצרנו את הנתונים בעצמנו, אפשר לצבוע את הנקודות לפי כמות אחד החומרים ולראות מה PCA גילתה.

In [ ]:
plt.figure()
scatter = plt.scatter(scores[:, 0], scores[:, 1], c=fractions[:, 0])
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA colored by fraction of A')
plt.colorbar(scatter, label='fraction of A')
plt.show()

In [ ]:
plt.figure()
scatter = plt.scatter(scores[:, 0], scores[:, 1], c=fractions[:, 1])
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA colored by fraction of B')
plt.colorbar(scatter, label='fraction of B')
plt.show()

הצבעים מראים שהצירים של PCA קשורים להרכב הכימי של התערובות, אף על פי שהחישוב עצמו קיבל רק את הספקטרים ולא את ההרכבים.

## איך נראים הצירים הספקטרליים?\n\nאפשר גם להציג את הכיוונים הראשונים של PCA כספקטרים. אלה אינם ספקטרים של חומרים טהורים, אלא תבניות שינוי חשובות בנתונים.

In [ ]:
pc1_loading = Vt[0]
pc2_loading = Vt[1]

plt.figure()
plt.plot(wavelength, pc1_loading, label='PC1 loading')
plt.plot(wavelength, pc2_loading, label='PC2 loading')
plt.xlabel('wavelength (nm)')
plt.ylabel('loading')
plt.title('PCA loadings')
plt.legend()
plt.show()

## תרגילים קצרים\n\n1. הגדילו את הרעש ובדקו האם גרף ה־PCA עדיין ברור.\n2. שנו את מרכזי הפסים של החומרים הטהורים. האם קל יותר או קשה יותר להפריד בין התערובות?\n3. צרו רק שתי משפחות של תערובות: אחת עשירה ב־A ואחת עשירה ב־C. האם רואים שתי קבוצות בגרף?\n4. נסו להציג את שלושת הרכיבי הראשונים במקום שניים.